# Milestone 4

This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model-ready inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.

---
---

In [3]:
import pandas as pd 
import numpy as np

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

In [4]:
train_df

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B


---

Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

---

### Question 1:


Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?


In [5]:
label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df['encoded_answer'] = train_df['answer'].map(label_mapping)
encoded_label_150 = train_df.loc[150, 'encoded_answer']
print(f"Encoded label at index 150: {encoded_label_150}")

Encoded label at index 150: 2


### Question 2:

Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [6]:
option_b_input = str(train_df.loc[0, "prompt"]) + " [SEP] " + str(train_df.loc[0, "B"])
option_b_length = len(option_b_input)
print(option_b_length)

407


---

Tokenization for Multiple-Choice Models
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

---

### Question 3:


Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?


In [7]:
from transformers import AutoTokenizer

row_idx = 0
choices = ["A", "B", "C", "D", "E"]

formatted_inputs = [
    str(train_df.loc[row_idx, "prompt"]) + " [SEP] " + str(train_df.loc[row_idx, choice])
    for choice in choices
]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

input_ids = encoded["input_ids"].unsqueeze(0)  # [1, 5, 128]
print(input_ids.shape[1])  # second dimension

5


### Question 4:

Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [8]:
batch_size = 16
batch_formatted_inputs = []

for idx in range(batch_size):
    for choice in choices:
        formatted_input = str(train_df.loc[idx, "prompt"]) + " [SEP] " + str(train_df.loc[idx, choice])
        batch_formatted_inputs.append(formatted_input)

batch_encoded = tokenizer(
    batch_formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

batch_input_ids = batch_encoded["input_ids"].view(batch_size, len(choices), 128)

total_tokens = batch_input_ids.shape[0] * batch_input_ids.shape[1] * batch_input_ids.shape[2]
print(f"Total token positions: {total_tokens}")

Total token positions: 10240


---

Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

---

### Question 5:

Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [9]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

mc_inputs = {
    "input_ids": encoded["input_ids"].unsqueeze(0),
    "token_type_ids": encoded["token_type_ids"].unsqueeze(0),
    "attention_mask": encoded["attention_mask"].unsqueeze(0),
}

outputs = model(**mc_inputs)
logits = outputs.logits  # shape [1, 5]
print(logits.shape[1])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2972.31it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

5


### Question 6:

Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [10]:
import torch

labels = torch.tensor([train_df.loc[0, "encoded_answer"]])
outputs_with_loss = model(**mc_inputs, labels=labels)
loss = outputs_with_loss.loss
print(loss.dim())

0


---

LoRA for Efficient Fine-Tuning
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

---

### Question 7:

 LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [11]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Trainable parameters: 295681


---

Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

---

### Question 8:

Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [16]:
from datasets import Dataset
import torch

def create_mcq_dataset(df, num_rows=100):
    dataset_dict = {
        'input_ids': [],
        'attention_mask': [],
        'labels': []
    }
    
    for idx in range(num_rows):
        formatted_inputs = [
            str(df.loc[idx, "prompt"]) + " [SEP] " + str(df.loc[idx, choice])
            for choice in choices
        ]
        
        encoded = tokenizer(
            formatted_inputs,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        
        dataset_dict['input_ids'].append(encoded['input_ids'])
        dataset_dict['attention_mask'].append(encoded['attention_mask'])
        dataset_dict['labels'].append(df.loc[idx, 'encoded_answer'])
    
    dataset_dict['input_ids'] = torch.stack(dataset_dict['input_ids'])
    dataset_dict['attention_mask'] = torch.stack(dataset_dict['attention_mask'])
    dataset_dict['labels'] = torch.tensor(dataset_dict['labels'])
    
    hf_dataset = Dataset.from_dict({
        'input_ids': dataset_dict['input_ids'],
        'attention_mask': dataset_dict['attention_mask'],
        'labels': dataset_dict['labels']
    })
    
    return hf_dataset


train_dataset = create_mcq_dataset(train_df, num_rows=100)

print(f"{len(train_dataset[0]['input_ids'])}")

5


---

Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

---

### Question 9:

 Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [17]:
from transformers import Trainer, TrainingArguments

# Build a tiny 32-row multiple-choice training dataset with max_length=64
train_examples_32 = []
for i in range(32):
    formatted_inputs = [
        str(train_df.loc[i, "prompt"]) + " [SEP] " + str(train_df.loc[i, c])
        for c in choices
    ]
    enc = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=64,
    )
    train_examples_32.append(
        {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc["token_type_ids"],
            "labels": int(train_df.loc[i, "encoded_answer"]),
        }
    )

train_dataset_32 = Dataset.from_list(train_examples_32)

def mc_data_collator(features):
    return {
        "input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
        "attention_mask": torch.tensor([f["attention_mask"] for f in features], dtype=torch.long),
        "token_type_ids": torch.tensor([f["token_type_ids"] for f in features], dtype=torch.long),
        "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long),
    }

training_args = TrainingArguments(
    output_dir="./tmp_lora_mcq",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_32,
    data_collator=mc_data_collator,
)

trainer.train()
print(trainer.state.global_step)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.533340
2,1.548349
3,1.687553
4,1.682317


4


### Question 10:

Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [19]:
import torch.nn.functional as F

row_idx = 0
formatted_inputs = [
    str(train_df.loc[row_idx, "prompt"]) + " [SEP] " + str(train_df.loc[row_idx, choice])
    for choice in choices
]

inf_encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt",
)
device = next(model.parameters()).device

inf_inputs = {
    "input_ids": inf_encoded["input_ids"].unsqueeze(0).to(device),
    "attention_mask": inf_encoded["attention_mask"].unsqueeze(0).to(device),
    "token_type_ids": inf_encoded["token_type_ids"].unsqueeze(0).to(device),
}


model.eval()
with torch.no_grad():
    inf_outputs = model(**inf_inputs)
    probs = F.softmax(inf_outputs.logits, dim=-1)

prob_option_e = probs[0, 4].item()
print(round(prob_option_e, 4))

0.1996
